# Loading Libraries

In [91]:
import io, os
import json
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage
from dotenv import load_dotenv

In [92]:
from langchain.tools import tool
from langchain.agents import create_agent
from langchain_openai import ChatOpenAI 
from langgraph.graph import StateGraph, END, START, add_messages
from typing import TypedDict, Annotated, Optional
from pydantic import BaseModel

In [93]:
from langsmith import traceable

# Loading the Model

In [94]:
load_dotenv()

True

In [95]:
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
OPEN_AI_MODEL = os.getenv("OPEN_AI_MODEL")

In [207]:
os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_PROJECT"] = "CodeReviewMAS_MessageCorruptedFault"

In [208]:
open_ai_model = ChatOpenAI(model_name=OPEN_AI_MODEL, openai_api_key=OPENAI_API_KEY, temperature=0, max_tokens=1000)

# Develop the agents

In [98]:
class State(TypedDict):
    pr_title: str
    pr_description: str
    file_path: str
    changed_code: str
    correctness: str
    security: str
    performance: str
    combined_output: str

In [99]:
TRIAGE_SYSTEM_PROMPT="""You are the Triage Agent in a multi-agent code review system.

Your role is to orchestrate the review process, not to review the code yourself.

Given a pull request, perform the following steps:

1. Understand the purpose of the code changes by reading the PR title, description, and the changed code.
2. Divide the review into three independent tasks and assign them to the following specialized agents:
   - Correctness Reviewer: Verify logic, functionality, and API usage.
   - Security Reviewer: Identify security vulnerabilities and unsafe coding practices.
   - Performance Reviewer: Detect performance bottlenecks and inefficient implementations.
Do not perform the technical review yourself or invent findings. Your responsibility is limited to task delegation.
"""

In [100]:
CORRECTNESS_SYSTEM_PROMPT="""You are the Correctness Reviewer in a multi-agent code review system.

Review the given code changes only for functional correctness. Identify:
- Logic errors and incorrect behavior
- API misuse
- Missing edge cases
- Exception handling issues
- Bugs introduced by the changes

For each issue, provide:
- Severity (Critical/High/Medium/Low)
- Location (file/line if available)
- Description
- Suggested fix

Keep the review comments concise.
Do not review security, performance, or code style. If no issues are found, state: "No correctness issues found.
"""

In [101]:
SECURITY_SYSTEM_PROMPT="""You are the Security Reviewer in a multi-agent code review system.

Review the code changes only for security issues. Identify:
- Vulnerabilities and unsafe practices
- Authentication and authorization flaws
- Input validation issues
- Injection risks
- Sensitive data exposure
- Insecure dependencies or configurations

For each issue, provide:
- Severity (Critical/High/Medium/Low)
- Location (file/line if available)
- Description
- Suggested fix

Keep the review comments concise.
Do not review correctness, performance, or code style. If no security issues are found, state: "No security issues found."""

In [102]:
PERFORMANCE_SYSTEM_PROMPT="""You are the Performance Reviewer in a multi-agent code review system.

Review the code changes only for performance issues. Identify:
- Inefficient algorithms or unnecessary computation
- Slow database, API, or I/O operations
- Memory inefficiencies or resource leaks
- Scalability bottlenecks
- Unnecessary duplication or repeated work

For each issue, provide:
- Severity (Critical/High/Medium/Low)
- Location (file/line if available)
- Description
- Suggested optimization

Keep the review comments concise.
Do not review correctness, security, or code style. If no performance issues are found, state: "No performance issues found.
"""

In [122]:
AGGREGATE_SYSTEM_PROMPT="""You are the Aggregator Agent in a multi-agent code review system.

Your role is to combine findings from specialized reviewers into a final review report.

Tasks:
- Collect and summarize findings from correctness, security, and performance reviewers.
- Remove duplicate findings.
- Highlight conflicting opinions.
- Prioritize issues by severity.
- Provide an overall recommendation:
  APPROVE, REQUEST CHANGES, or COMMENT ONLY.

Do not introduce new findings or perform your own code analysis. Base the report only on reviewer feedback.
"""

In [217]:
@traceable(
    run_type="chain",
    name="Triage agent"
)
def triage_code_review(state: State):
    """Coordinate the code review task and assign to appropriate reviewers."""
    response = open_ai_model.invoke([SystemMessage(content=TRIAGE_SYSTEM_PROMPT), 
                                     HumanMessage(content=f"PR title: {state['pr_title']}\nPR description: {state['pr_description']}\nChanged code: {state['changed_code']}")
    ])
    return state

In [218]:
@traceable(
    run_type="chain",
    name="Correctness Reviewer"
)
def review_correctness(state: State):
    """Check the correctness of the changed code."""
    response = open_ai_model.invoke([SystemMessage(content=CORRECTNESS_SYSTEM_PROMPT), 
                                     HumanMessage(content=f"Changed code: {state['changed_code']}")])
    return {"correctness": response.content}

In [219]:
@traceable(
    run_type="chain",
    name="Security Reviewer"
)
def review_security_issues(state: State):
    """Check for security vulnerabilities in the changed code."""
    response = open_ai_model.invoke([SystemMessage(content=SECURITY_SYSTEM_PROMPT), 
                                     HumanMessage(content=f"Changed code: {state['changed_code']}")])
    return {"security": response.content}

In [220]:
@traceable(
    run_type="chain",
    name="Performance Reviewer"
)
def review_performance(state: State):
    """Check for performance issues in the changed code."""
    response = open_ai_model.invoke([SystemMessage(content=PERFORMANCE_SYSTEM_PROMPT), 
                                     HumanMessage(content=f"Changed code: {state['changed_code']}")])
    return {"performance":response.content} 

In [221]:
@traceable(
    run_type="chain",
    name="Aggregator Agent"
)
def aggregate_findings(state: State):
    """Aggregate the findings from the reviewers."""
    response = open_ai_model.invoke([SystemMessage(content=AGGREGATE_SYSTEM_PROMPT),        
                                     HumanMessage(content=f"Correctness findings: {state["correctness"]}\nSecurity findings: {state["security"]}\nPerformance findings: {state["performance"]}")])
    return {"combined_output": response.content}

In [222]:
import random

def lose_half_of_string(s: str, seed: int | None = None) -> str:
    """Return a string with roughly 50% of characters removed."""
    if seed is not None:
        random.seed(seed)
    n = len(s)
    if n == 0:
        return s
    keep_count = n // 2
    keep_indices = set(random.sample(range(n), keep_count))
    return "".join(s[i] for i in range(n) if i in keep_indices)

In [223]:
def lose_findings(state: State):
    correctness = state["correctness"]
    state["correctness"] = lose_half_of_string(correctness,100)
    security = state["security"]
    state["security"] = lose_half_of_string(security,100)
    performance = state["performance"]
    state["performance"] = lose_half_of_string(performance,100)
    return state

In [224]:
@traceable(
    run_type="chain",
    name="Aggregator Agent"
)
def aggregate_findings_message_lost(state: State):
    state=lose_findings(state)
    response = open_ai_model.invoke([SystemMessage(content=AGGREGATE_SYSTEM_PROMPT),        
                                     HumanMessage(content=f"Correctness findings: {state["correctness"]}\nSecurity findings: {state["security"]}\nPerformance findings: {state["performance"]}")])
    return {"combined_output": response.content}

In [225]:
def perform_rule_based_corruption(message):
    replacements=["critical", "medium", "high"]
    for key in replacements:
        message = message.lower().replace(key, "low")
    message = message.replace("no", "many")
    message = message.replace("location", "no")
    return message

In [226]:
def corrupt_findings(state: State):
    correctness = state["correctness"]
    state["correctness"] = perform_rule_based_corruption(correctness)
    security = state["security"]
    state["security"] = perform_rule_based_corruption(security)
    performance = state["performance"]
    state["performance"] = perform_rule_based_corruption(performance)
    return state

In [227]:
@traceable(
    run_type="chain",
    name="Aggregator Agent"
)
def aggregate_findings_message_corrupted(state: State):
    state=corrupt_findings(state)
    response = open_ai_model.invoke([SystemMessage(content=AGGREGATE_SYSTEM_PROMPT),        
                                     HumanMessage(content=f"Correctness findings: {state["correctness"]}\nSecurity findings: {state["security"]}\nPerformance findings: {state["performance"]}")])
    return {"combined_output": response.content}

# Develop the LangGraph

In [128]:
def devlop_compiled_network():
    code_review_network = StateGraph(State)
    code_review_network.add_node("triage", triage_code_review)
    code_review_network.add_node("correctness", review_correctness)
    code_review_network.add_node("security", review_security_issues)
    code_review_network.add_node("performance", review_performance)
    code_review_network.add_node("aggregate", aggregate_findings)

    code_review_network.add_edge(START, "triage")
    code_review_network.add_edge("triage", "correctness")
    code_review_network.add_edge("triage", "security")
    code_review_network.add_edge("triage", "performance")
    code_review_network.add_edge("correctness", "aggregate")
    code_review_network.add_edge("security", "aggregate")
    code_review_network.add_edge("performance", "aggregate")
    code_review_network.add_edge("aggregate", END)

    return code_review_network.compile()   

In [129]:
def devlop_compiled_network_with_delegation_fault():
    code_review_network = StateGraph(State)
    code_review_network.add_node("triage", triage_code_review)
    
    # All three delegations are swapped - fault injected
    code_review_network.add_node("correctness", review_security_issues)
    code_review_network.add_node("security", review_performance)
    code_review_network.add_node("performance", review_correctness)
    code_review_network.add_node("aggregate", aggregate_findings)
    
    code_review_network.add_edge(START, "triage")
    code_review_network.add_edge("triage", "correctness")
    code_review_network.add_edge("triage", "security")
    code_review_network.add_edge("triage", "performance")
    code_review_network.add_edge("correctness", "aggregate")
    code_review_network.add_edge("security", "aggregate")
    code_review_network.add_edge("performance", "aggregate")
    code_review_network.add_edge("aggregate", END)

    return code_review_network.compile()

In [189]:
def devlop_compiled_network_with_message_loss_fault():
    code_review_network = StateGraph(State)
    code_review_network.add_node("triage", triage_code_review)
    code_review_network.add_node("correctness", review_correctness)
    code_review_network.add_node("security", review_security_issues)
    code_review_network.add_node("performance", review_performance)
    # Simulating message lost before aggregation
    code_review_network.add_node("aggregate", aggregate_findings_message_lost)

    code_review_network.add_edge(START, "triage")
    code_review_network.add_edge("triage", "correctness")
    code_review_network.add_edge("triage", "security")
    code_review_network.add_edge("triage", "performance")
    code_review_network.add_edge("correctness", "aggregate")
    code_review_network.add_edge("security", "aggregate")
    code_review_network.add_edge("performance", "aggregate")
    code_review_network.add_edge("aggregate", END)

    return code_review_network.compile()

In [231]:
def devlop_compiled_network_with_corrupted_message():
    code_review_network = StateGraph(State)
    code_review_network.add_node("triage", triage_code_review)
    code_review_network.add_node("correctness", review_correctness)
    code_review_network.add_node("security", review_security_issues)
    code_review_network.add_node("performance", review_performance)
    # Simulating message corruption before aggregation
    code_review_network.add_node("aggregate", aggregate_findings_message_corrupted)

    code_review_network.add_edge(START, "triage")
    code_review_network.add_edge("triage", "correctness")
    code_review_network.add_edge("triage", "security")
    code_review_network.add_edge("triage", "performance")
    code_review_network.add_edge("correctness", "aggregate")
    code_review_network.add_edge("security", "aggregate")
    code_review_network.add_edge("performance", "aggregate")
    code_review_network.add_edge("aggregate", END)

    return code_review_network.compile()

In [232]:
code_review_mas=devlop_compiled_network_with_corrupted_message()

# Diagnosis & Testing

In [233]:
pr_title="Avoid accessing non-existing \"$ref\" key for Pydantic v2 compat remapping"
pr_description="cf Discussion #14265"
review_comment="For my case reported in https://github.com/fastapi/fastapi/discussions/14265 I am able to successfully generate an OpenAPI specification file without erroring out when using this branch of FastAPI.\r\n\r\nThanks!\nThanks for reporting back, @jerry-reevo !\nJust adding in, my team was also facing the same issue discussed in this PR which has kept us stuck at `fastapi==0.118.3`. This branch has seemed to resolved this issue for me (at least for local testing).\nI can also confirm this PR fixes the issue for me. Thanks @svlandeg!"
changed_code="old_name_to_new_name_map = {}\n    for field_key, schema in field_mapping.items():\n        model = field_key[0].type_\n        if model not in model_name_map:\n        if model not in model_name_map or \"$ref\" not in schema:\n            continue\n        new_name = model_name_map[model]\n        old_name = schema[\"$ref\"].split(\"/\")[-1]"

In [234]:
import langsmith as ls

# Review the Code

In [235]:
def load_repo_data(repo_path):
    with open(repo_path, "r") as f:
        return f.read()

In [236]:
def load_code_changes(json_content):
   return json.loads(json_content)

In [237]:
def review_code_from_pr(element, ls_config, ls_project_name="CodeReviewMAS"):
    if element["changed_code"] is not None:
        pr_title, pr_description=element["pr_title"], element["pr_description"]
        changed_code=element["changed_code"]
        with ls.tracing_context(enable=True, project_name=ls_project_name, metadata=ls_config["metadata"]):
            review_comment = code_review_mas.invoke({"pr_title": pr_title, "pr_description": pr_description, "changed_code": changed_code}, config=ls_config)
        ground_truth = element["review_comments"]
        return {"pr_number": element["pr_number"], "repo": element["repo"], "file_path": element["file_path"], 
                "review_comment": review_comment["combined_output"], "ground_truth": ground_truth}

In [238]:
def make_ls_config(repo_name, first_element):
    ls_config = {
        "metadata": {
            "repository": repo_name,
            "pr_number": first_element["pr_number"],
            "repo": first_element["repo"],
            "file_path": first_element["file_path"]
        }
    }
    return ls_config

In [239]:
def review_and_save_from_pr(element, ls_config, faulty_model_name):
    evaluated_result=review_code_from_pr(element, ls_config, faulty_model_name)
    return evaluated_result

In [240]:
import time

In [241]:
def save_outputs(out_file, saved_outputs):
    with open(out_file, "w") as f:
        json.dump(saved_outputs, f)

In [242]:
def run_experiment(repo_name, elements, out_file, faulty_model_name):
    outputs=[]
    for element in elements:
        ls_config = make_ls_config(repo_name, element)
        evaluated_result=review_and_save_from_pr(element, ls_config, faulty_model_name)
        outputs.append(evaluated_result)
        time.sleep(1)
        
    save_outputs(out_file, outputs)

In [249]:
def get_elements(repo_path):
    json_repo_data=load_repo_data(repo_path)
    elements=load_code_changes(json_repo_data)
    return elements

In [250]:
repo_name="pandas"
repo_path="dataset/processed/pandas.json"
output_path="dataset/faulty/pandas-message-corrupted-fault.json"
faulty_model_name="CodeReviewMAS_MessageCorruptedFault"
elements=get_elements(repo_path)
len(elements)

14

In [251]:
run_experiment(repo_name, elements, output_path, faulty_model_name)